# **Project Name**    -  Shopper Spectrum: Customer Segmentation and Product Recommendations in E-Commerce


##### **Project Type**    - Unsupervised
##### **Contribution**    - Individual

# **Project Summary -**

The e-commerce industry generates enormous volumes of transactional data every day, and businesses that can extract meaningful patterns from this data gain a significant competitive advantage. This project, Shopper Spectrum, focuses on analysing real-world online retail transaction data to understand customer purchasing behaviour, segment customers into meaningful groups, and build a product recommendation system.

The dataset used in this project is the Online Retail dataset, containing 541,909 transactions across 38 countries, spanning product purchases made by thousands of unique customers. Each record includes the invoice number, stock code, product description, quantity purchased, invoice date, unit price, customer ID, and country. Before any analysis could be performed, the dataset required significant cleaning. A new feature, TotalAmount, was engineered by multiplying Quantity and UnitPrice to represent the monetary value of each transaction.

The core of the project is built around RFM (Recency, Frequency, Monetary) analysis. Recency measures how recently a customer made a purchase, Frequency counts how many transactions they completed, and Monetary captures the total amount they have spent. These three features were computed for every unique customer in the dataset and then standardised using StandardScaler to eliminate scale bias before applying clustering algorithms.

Three unsupervised clustering algorithms were evaluated: KMeans, DBSCAN, and Hierarchical (Agglomerative) Clustering. The optimal number of clusters was determined using the Elbow Method and validated using Silhouette Scores. KMeans achieved the best interpretable silhouette score of 0.6165 with 5 clusters, and Hierarchical Clustering confirmed the same with a score of 0.6085. DBSCAN, despite showing a high silhouette score of 0.8634, was rejected because it classified high value customers as outliers rather than assigning them to meaningful segments. KMeans was selected as the final model.


The five customer segments identified are: VIP Customers (recent, frequent, high spenders), Loyal Customers (consistent purchasers with strong engagement), Regular Customers (moderate activity across all RFM dimensions), Occasional Customers (infrequent and low-value buyers), and At-Risk Customers (previously active customers showing declining engagement).

In addition to segmentation, a collaborative filtering-based product recommendation system was built using cosine similarity on a customer-product purchase matrix. For any given product, the system identifies the top 5 most similar products based on co-purchase patterns across customers.

The entire pipeline was deployed as an interactive Streamlit web application, allowing users to input RFM values to predict their customer segment, or enter a product name to receive personalised product recommendations. The application loads pre-trained models saved via joblib, making it lightweight and production-ready.


# **Problem Statement**


The global e-commerce industry generates vast amounts of transaction data daily, offering valuable insights into customer purchasing behaviors. Analyzing this data is essential for identifying meaningful customer segments and recommending relevant products to enhance customer experience and drive business growth. This project aims to examine transaction data from an online retail business to uncover patterns in customer purchase behavior, segment customers based on Recency, Frequency, and Monetary (RFM) analysis, and develop a product recommendation system using collaborative filtering techniques.

# **General Guidelines** : -  

1.   Well-structured, formatted, and commented code is required.
2.   Exception Handling, Production Grade Code & Deployment Ready Code will be a plus. Those students will be awarded some additional credits.
     
     The additional credits will have advantages over other students during Star Student selection.
       
             [ Note: - Deployment Ready Code is defined as, the whole .ipynb notebook should be executable in one go
                       without a single error logged. ]

3.   Each and every logic should have proper comments.
4. You may add as many number of charts you want. Make Sure for each and every chart the following format should be answered.
        

```
# Chart visualization code
```
            

*   Why did you pick the specific chart?
*   What is/are the insight(s) found from the chart?
* Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

5. You have to create at least 15 logical & meaningful charts having important insights.


[ Hints : - Do the Vizualization in  a structured way while following "UBM" Rule.

U - Univariate Analysis,

B - Bivariate Analysis (Numerical - Categorical, Numerical - Numerical, Categorical - Categorical)

M - Multivariate Analysis
 ]





6. You may add more ml algorithms for model creation. Make sure for each and every algorithm, the following format should be answered.


*   Explain the ML Model used and it's performance using Evaluation metric Score Chart.


*   Cross- Validation & Hyperparameter Tuning

*   Have you seen any improvement? Note down the improvement with updates Evaluation metric Score Chart.

*   Explain each evaluation metric's indication towards business and the business impact pf the ML model used.




















# ***Let's Begin !***

## ***1. Know Your Data***

### Import Libraries

In [ ]:
# Import Libraries
import pandas as pd
import matplotlib.pyplot as plt
import calendar
import seaborn as sns
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans,DBSCAN,AgglomerativeClustering
from scipy.cluster.hierarchy import linkage,dendrogram
from sklearn.metrics import silhouette_score
import joblib
from sklearn.metrics.pairwise import cosine_similarity
from google.colab import files


### Dataset Loading

In [ ]:
# Load Dataset
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
df = pd.read_csv("/content/drive/MyDrive/Shopper Spectrum/online_retail.csv")

### Dataset First View

In [ ]:
# Dataset First Look
df.info()

### Dataset Rows & Columns count

In [ ]:
# Dataset Rows & Columns count
print(df.shape)
print(f"The number of rows and columns are {df.shape[0]} and {df.shape[1]} respectively.")

### Dataset Information

#### Duplicate Values

In [ ]:
# Dataset Duplicate Value Count
print(f"The number of duplicates present are {df.duplicated().sum()}.")

#### Missing Values/Null Values

In [ ]:
# Missing Values/Null Values Count
null_count = df.isnull().sum()
print(f"The number of missing/ null value are:\n{null_count}")

In [ ]:
# Visualizing the missing values
plt.figure()
plt.title("Null Values Visualization", fontsize = 15)
plt.xlabel("Categories")
plt.ylabel("Count")
bars = plt.bar(null_count.index, null_count.values)
for x,y in enumerate(null_count):
  plt.text(x,y, str(y),ha="center",va="bottom")
plt.xticks(rotation =45 )
plt.show()


### What did you know about your dataset?


From the given dataset, it is evident that it has 8 cloumns and 541909 entries.

The dataset has three data types: float64, object, int64.

There are 135080 and 1454 null values presemt in CustomerId and description respectively.

The number of duplicates present are 5268.

The given dataset is not clean and request preprocessing.

## ***2. Understanding Your Variables***

In [ ]:
# Dataset Columns
print(f"The columns present in the dataset are:\n{df.columns}")

In [ ]:
# Dataset Describe
print(df.describe())

### Variables Description

**InvoiceNo**: Unique transaction identifier.

**StockCode:** Unique product identifier.

**Description:** Product name.

**Quantity:** Number of units purchased.

**InvoiceDate:** Transaction timestamp.

**UnitPrice:** Price per unit.

**CustomerID:** Unique customer identifier.


**Country:** Customer location.

### Check Unique Values for each variable.

In [ ]:
# Check Unique Values for each variable.
print(df.nunique())

## 3. ***Data Wrangling***

### Data Wrangling Code

In [ ]:
# Write your code to make your dataset analysis ready.
df_clean = df.copy()

#### Remove Duplicates

In [ ]:
df_clean.drop_duplicates(inplace=True)
print(df_clean.shape)
print(f"The number of duplicates are:\n{df_clean.duplicated().sum()}")

#### Remove Missing Description

In [ ]:
df_clean = df_clean.dropna(subset=['Description'])
print(f"The count of missing values present is: \n{df_clean.isnull().sum()} ")

#### Remove Missing values from CustomerID


In [ ]:
df_clean = df_clean.dropna(subset=['CustomerID'])
print(f"The count of missing values present is: \n{df_clean.isnull().sum()} ")

#### Exclude cancelled invoices (InvoiceNo starting with 'C')

In [ ]:
df_clean[df_clean['InvoiceNo'].astype(str).str.startswith('C')].shape

In [ ]:
df_clean = df_clean[~df_clean['InvoiceNo'].astype(str).str.startswith('C')]

#### Remove negative or zero quantities and prices


In [ ]:
print((df_clean['UnitPrice']<=0).sum())
print((df_clean['Quantity']<=0).sum())

In [ ]:
df_clean = df_clean[df_clean["UnitPrice"]>0]
print(f"Negative or zero UnitPrice:{(df_clean['UnitPrice']<=0).sum()}")
df_clean = df_clean[df_clean["Quantity"]>0]
print(f"negative or zero quantities:{(df_clean['Quantity']<=0).sum()}")

#### Convert Data Types

In [ ]:
df_clean['InvoiceDate'] = pd.to_datetime(df_clean['InvoiceDate'],dayfirst=True)
df_clean['CustomerID'] = df_clean['CustomerID'].astype(int)

In [ ]:
df_clean.info()

#### Creation of Sales Features

In [ ]:
df_clean['TotalAmount'] = (df_clean['Quantity']*df_clean['UnitPrice'])

In [ ]:
df_clean.head(5)

### What all manipulations have you done and insights you found?

1. 5268 duplicate records were removed.
2. Removed records with missing CustomerId and Description.
3. Excluded cancelled invoices starting with "C".
4. Removed Invaild values of quantites and unitprice having negative or zero values.
5. Converted CustomerID to integer format.
6. Converted InvoiceDate to datetime format.
7. Created a new feature totalamount representing total transaction value.

## ***4. Exploratory Data Analysis(EDA)***

### Analyze transaction volume by country

In [ ]:
country_sales = df_clean.groupby('Country').count()['InvoiceNo'].sort_values(ascending=False)
print(country_sales)
print(df_clean.head(5))

In [ ]:
plt.figure()
plt.title("Top 10 Countries by Transaction Volume", fontsize=18)
plt.xlabel('Country', fontsize=14)
plt.ylabel('Transactions', fontsize=14)
plt.xticks(rotation=45)
plt.bar(country_sales.head(10).index, country_sales.head(10).values)


The United Kingdom contributes the highest transaction volume, indicating it is the primary market for the business.

### Top-selling products

In [ ]:
top_products = df_clean.groupby('Description')['Quantity'].sum().sort_values(ascending =False)
print(top_products.head(10))

In [ ]:
plt.figure()
plt.title('Top Selling Products', fontsize=18)
plt.xlabel('Description', fontsize=14)
plt.ylabel('Quantity', fontsize=14)
plt.xticks(rotation= 90)
plt.bar(top_products.head(10).index, top_products.head(10).values)

Paper Craft, Little birdie was the most purchased item with a total count of 80995.

### Revenue By Country


In [ ]:
revenue_country = df_clean.groupby('Country')['TotalAmount'].sum().sort_values(ascending = False)

In [ ]:
plt.figure()
plt.title('Revenue By Country', fontsize=18)
plt.xlabel('Country', fontsize=14)
plt.ylabel('Revenue', fontsize=14)
plt.xticks(rotation= 45)
plt.bar(revenue_country.head(10).index, revenue_country.head(10).values)

###	Monthly Sales Trend


In [ ]:
monthly_sales = df_clean.groupby(df_clean['InvoiceDate'].dt.month)['TotalAmount'].sum()

In [ ]:
plt.figure()
plt.title('Monthly Revenue Trend', fontsize=18)
plt.xlabel('Month', fontsize=14)
plt.ylabel('Revenue', fontsize=14)
monthly_sales.index = monthly_sales.index.map(lambda x:calendar.month_abbr[x])
plt.bar(monthly_sales.index, monthly_sales.values)

From the graph, the month of November has the highest sales followed by December with February having the lowest revenue.

### Monetary distribution per transaction

In [ ]:
plt.figure()
sns.histplot(np.log1p(df_clean['TotalAmount']), bins = 100)
plt.title('Monetary distribution per transaction', fontsize=18)
plt.xlabel('Log(Transacion Amount)', fontsize=14)
plt.ylabel('Frequency', fontsize=14)
plt.show()

The monetary distribution per transaction is strongly skewed to the positive. The majority of transactions have relatively low values, but a tiny number of transactions generate very large buy amounts. This suggests the occurrence of outliers and high-value purchases, which are prevalent in e-commerce databases.


### Monetary distribution per customer

In [ ]:
customer_monetary = df_clean.groupby('CustomerID')['TotalAmount'].sum()

In [ ]:
plt.figure()
sns.histplot(np.log1p(customer_monetary), bins = 100)
plt.title('Monetary distribution per customer', fontsize=18)
plt.xlabel('Log(Transacion Amount)', fontsize=14)
plt.ylabel('Frequency', fontsize=14)
plt.show()

## ***5. Feature Engineering***

### RFM distributions

In [ ]:
newest_date = df_clean['InvoiceDate'].max() + pd.Timedelta(days=1)

In [ ]:
rfm = df_clean.groupby('CustomerID').agg({"InvoiceDate": lambda x:(newest_date-x.max()).days,
                                          'InvoiceNo':'nunique',
                                          'TotalAmount':'sum'})
rfm.columns=['Recency','Frequency','Monetary']

In [ ]:
plt.figure()
plt.hist(rfm['Recency'], bins=80, color= 'orange')
plt.title("RFM Distribution - Recency")
plt.xlabel('Number of Days')
plt.ylabel('Number of Customers')
plt.show()
print('\n')

plt.figure()
plt.hist(rfm['Frequency'], bins=100, color= 'orange')
plt.title("RFM Distribution - Frequency")
plt.xlabel('Number of Purchases')
plt.ylabel('Number of Customers')
plt.show()
print('\n')

plt.figure()
plt.hist(rfm['Monetary'], bins=100, color= 'orange')
plt.title("RFM Distribution - Monetary")
plt.xlabel('Amount Spent')
plt.ylabel('Number of Customers')
plt.show()

## ***6. Data Scaling***

In [ ]:
scaler = StandardScaler()
rfm_scaled =scaler.fit_transform(rfm[['Recency','Frequency','Monetary']])

## ***7. ML Model Implementation***

### KMeans

#### Elbow Method

In [ ]:
inertia=[]
for k in range(1,11):
  kmeans = KMeans(n_clusters=k,random_state=42)
  kmeans.fit(rfm_scaled)
  inertia.append(kmeans.inertia_)

In [ ]:
plt.plot(range(1,11),inertia,marker='x')
plt.xlabel('Number of Clusters')
plt.ylabel('Inertia')

#### Silhouette Score

In [ ]:
kmeans_scores=[]
for k in range(2,11):
  kmeans = KMeans(n_clusters=k, random_state=42)
  predict = kmeans.fit_predict(rfm_scaled)
  kmeans_score = silhouette_score(rfm_scaled, predict)
  kmeans_scores.append(kmeans_score)
  print(f"k={k}, Silhouette Score = {kmeans_score}")

In [ ]:
plt.figure()
plt.plot(range(2,11),kmeans_scores,marker='o')
plt.title('Silhouette Scores for different values of k', fontsize=14)
plt.xlabel('Number of Clusters', fontsize=12)
plt.ylabel('Silhouette Score', fontsize=12)
plt.grid(True)

The Elbow methid indicated 4 clusters and the silhouette score was highest for k=4.
Hence, the number of clusters selected were 4.

### DBSCAN

In [ ]:
dbscan_scores=[]
cluster_counts=[]
noise_counts=[]
eps_values=[0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8]
for eps in eps_values:
  dbscan = DBSCAN(eps=eps, min_samples=5)
  clusters = dbscan.fit_predict(rfm_scaled)
  dbscan_score=silhouette_score(rfm_scaled,clusters)
  cluster_count= len([i for i in set(clusters) if i!=-1])
  noise_count = list(clusters).count(-1)
  print(f"eps = {eps}, Silhouette Score = {dbscan_score}")
  dbscan_scores.append(dbscan_score)
  cluster_counts.append(cluster_count)
  noise_counts.append(noise_count)


In [ ]:
plt.plot(eps_values,dbscan_scores, marker='o', label = 'Silehouette Score')
plt.plot(eps_values,cluster_counts,marker='x', label = 'Cluster Count')
plt.plot(eps_values,noise_counts,marker='*', label = 'Noise Count')
plt.legend()
plt.title('DBSCA Metrics vs EPS')
plt.ylabel('Value')
plt.xlabel('EPS')
plt.grid(True)
print('\n')


### Hierarchial

In [ ]:
linkage_model = linkage(rfm_scaled, method='ward')

In [ ]:
plt.figure()
plt.xlabel('Clusters')
plt.ylabel('Distance')
dendrogram(linkage_model,p=30,truncate_mode='lastp',leaf_rotation=50,leaf_font_size=9)

In [ ]:
hc_scores=[]
for n in [3,4,5]:
  model = AgglomerativeClustering(n_clusters=n)
  labels=model.fit_predict(rfm_scaled)
  hc_score = silhouette_score(rfm_scaled,labels)
  print(f"Clusters = {n}, Silhouette Score = {hc_score}")
  hc_scores.append(hc_score)

In [ ]:
plt.figure()
plt.plot([3,4,5],hc_scores, marker='o')
plt.title('Hierarchial Clustering Silhouette Scores')
plt.xlabel('Number of Clusters')
plt.ylabel('Silhouette Score')
plt.grid(True)

### Comparision

In [ ]:
comparision_df = pd.DataFrame({"Algorithm":['KMeans','DBSCAN','Hierarchial'],
                               "Best Parameter":[f'K={np.argmax(kmeans_scores)+2}',
                                                 f'eps={eps_values[np.argmax(dbscan_scores)]}',
                                                  f'n_clusters={[3,4,5][np.argmax(hc_scores)]}'],
                               'Best Silhouette Score': [max(kmeans_scores),
                                                         max(dbscan_scores),
                                                         max(hc_scores)]})
print(comparision_df)

From the comparison data frame, it is evident that k means and hierarchical algorithms have they have Silhouette score of 0.616500 and 0.608547respectively. And DBSCAN has Silhouette score of 0.863425. So we won't select the dbscan because  it it does not include all the data and some of the customers which are which are having higher purchase values, it identifies them as outliers and removes them. Also, it is mainly used in identifying spam mails and examples like that.
K means and hierarchical algorithms have the number of clusters as five as their best parameter. So we will use the same value for the number of clusters.



In [ ]:
kmeans_model = KMeans(n_clusters=5, random_state=42)
rfm['KMeans_Cluster'] = kmeans_model.fit_predict(rfm_scaled)
rfm.head(5)

## ***8. Cluster Interpretation***

### Customer cluster profiles

In [ ]:
cluster_customer = rfm.groupby('KMeans_Cluster')[['Recency','Frequency','Monetary']].mean()
cluster_customer

In [ ]:
customer_segments= {
    0: 'Occassional Customer',
    1: 'At risk customer',
    2: 'Loyal customer',
    3: 'Regular customer',
    4: 'VIP customer'
}

## ***9. Product Recommendation***

In [ ]:
customer_product = pd.pivot_table(data = df_clean,
                                  index = 'CustomerID',
                                  columns='Description',
                                  values= 'Quantity',
                                  fill_value=0
                                  )
customer_product.head(5)

In [ ]:
product_similarity = cosine_similarity(customer_product.T)
similarity_df = pd.DataFrame(
    data = product_similarity,
    index= customer_product.columns,
    columns = customer_product.columns
)
similarity_df.head(5)

In [ ]:
plt.figure()
sns.heatmap(similarity_df.iloc[:20, :20], cmap= 'coolwarm')
plt.title('Product Similarity Heatmap')
plt.xlabel('Products')
plt.ylabel('Products')

In [ ]:
def recommend_products(product_name, n=5):
  if product_name not in similarity_df.columns:
    return "Product not found"
  recommendations = (similarity_df[product_name]
                     .sort_values(ascending=False)
                     .iloc[1:n+1])
  return recommendations.index.tolist()

In [ ]:
recommend_products('WHITE HANGING HEART T-LIGHT HOLDER')

## ***10. Saving the model***

In [ ]:
kmeans_model = KMeans(n_clusters=5, random_state=42)
kmeans_model.fit(rfm_scaled)
joblib.dump(kmeans_model, 'kmeans_model.pkl')
joblib.dump(scaler, 'scaler.pkl')
joblib.dump(similarity_df, 'product_similarity.pkl' )

In [ ]:
files.download('kmeans_model.pkl')
files.download('scaler.pkl')
files.download('product_similarity.pkl' )

# **Conclusion**

This project analyzed customer purchasing behavior using e-commerce transaction data. After data cleaning and preprocessing, RFM (Recency, Frequency, and Monetary) features were generated to understand customer engagement and spending patterns. Exploratory Data Analysis (EDA) provided insights into sales trends, top-selling products, and customer behavior.

Customer segmentation was performed using K-Means clustering, resulting in meaningful customer groups that can support targeted marketing and retention strategies. Additionally, an item-based collaborative filtering recommendation system was developed using cosine similarity to suggest relevant products based on purchase history.

Overall, the project demonstrates how data analytics and machine learning techniques can be used to gain valuable business insights, improve customer understanding, and support data-driven decision-making in the e-commerce domain.


### ***Hurrah! You have successfully completed your Machine Learning Capstone Project !!!***